# TechArena 2025 Phase 2: Electricity Market Analysis
## McKinsey-Style Data Visualization

**Purpose**: Comprehensive analysis of European electricity market data across 4 markets:
- Day-Ahead Energy Market (15-min, EUR/MWh)
- FCR Capacity Market (4-hour blocks, EUR/MW)
- aFRR Capacity Market (4-hour blocks, EUR/MW, Pos/Neg)
- aFRR Energy Market (15-min, EUR/MWh, Pos/Neg)

**Countries**: Germany (DE), Austria (AT), Switzerland (CH), Hungary (HU), Czech Republic (CZ)

---

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add py_script to path
sys.path.insert(0, str(Path.cwd() / 'py_script'))

# Import custom modules
from data.market_data import (
    # McKinsey plotting functions
    plot_price_time_series_mckinsey,
    plot_da_price_distribution_mckinsey,
    plot_da_price_heatmap_mckinsey,
    calculate_price_statistics_mckinsey,
    plot_price_statistics_mckinsey,
    # Basic plotting functions
    plot_day_ahead_distribution,
    plot_fcr_distribution,
    plot_afrr_distribution,
)
from visualization.config import (
    MCKINSEY_COLORS,
    COUNTRY_COLORS,
    apply_mckinsey_style,
    get_country_color
)

# Configure Plotly to use McKinsey template
pio.templates.default = "mckinsey"

print("✓ Libraries imported successfully")
print(f"✓ McKinsey template activated")

ModuleNotFoundError: No module named 'py_script'

In [2]:
# Load processed Phase 2 market data from parquet files
data_dir = Path('data/phase2_processed/parquet')

print("Loading market data from parquet files...\n")

tables = {}
for market_file in data_dir.glob('*.parquet'):
    market_name = market_file.stem
    tables[market_name] = pd.read_parquet(market_file)
    print(f"✓ {market_name:20s}: {len(tables[market_name]):,} rows x {len(tables[market_name].columns)} columns")

# Load metadata
with open('data/phase2_processed/metadata.json', 'r') as f:
    import json
    metadata = json.load(f)

print(f"\n📅 Data Time Range: {metadata['date_range']['start'][:10]} to {metadata['date_range']['end'][:10]}")
print(f"📊 Processing Date: {metadata['processing_timestamp'][:10]}")

Loading market data from parquet files...

✓ afrr_capacity       : 2,197 rows x 11 columns
✓ afrr_energy         : 35,136 rows x 11 columns
✓ day_ahead           : 35,136 rows x 6 columns
✓ fcr                 : 2,197 rows x 6 columns

📅 Data Time Range: 2024-01-01 to 2024-12-31
📊 Processing Date: 2025-10-27


In [3]:
# Quick data preview
print("Data Preview - Day-Ahead Market:\n")
display(tables['day_ahead'].head(10))

print("\nData Preview - aFRR Energy Market:\n")
display(tables['afrr_energy'].head(10))

Data Preview - Day-Ahead Market:



,timestamp,DE_LU,AT,CH,HU,CZ
0,2024-01-01 00:00:00.000,39.91,14.08,25.97,0.10,0.10
1,2024-01-01 00:15:00.000,-0.04,14.08,25.97,0.10,0.10
2,2024-01-01 00:30:00.001,-9.01,0.48,25.97,0.10,0.10
3,2024-01-01 00:45:00.001,-29.91,-3.64,25.97,0.10,0.10
4,2024-01-01 01:00:00.001,25.28,17.74,21.99,0.01,0.01
5,2024-01-01 01:15:00.001,-24.91,14.94,21.99,0.01,0.01
6,2024-01-01 01:30:00.002,-0.07,-2.74,21.99,0.01,0.01
7,2024-01-01 01:45:00.002,-0.08,-14.91,21.99,0.01,0.01
8,2024-01-01 02:00:00.002,-0.05,15.09,14.32,0.04,0.02
9,2024-01-01 02:15:00.003,-0.06,9.98,14.32,0.04,0.02



Data Preview - aFRR Energy Market:



,timestamp,DE_Pos,DE_Neg,AT_Pos,AT_Neg,CH_Pos,CH_Neg,HU_Pos,HU_Neg,CZ_Pos,CZ_Neg
0,2024-01-01 00:00:00.000,50.341149,29.702030,86.43,0.00,38.70,0.00,0.0,0.000000,144.00,0.00
1,2024-01-01 00:15:00.000,46.945714,40.871250,85.25,0.17,38.80,8.32,0.0,0.601402,117.26,53.39
2,2024-01-01 00:30:00.001,43.874872,21.239111,85.44,0.73,38.76,0.00,0.0,1.133452,140.84,62.84
3,2024-01-01 00:45:00.001,0.000000,0.000000,84.29,0.86,509.00,0.00,0.0,0.000000,127.10,40.73
4,2024-01-01 01:00:00.001,83.175381,23.045000,84.52,0.55,41.79,0.00,0.0,0.169887,138.57,32.20
5,2024-01-01 01:15:00.001,61.604307,19.698902,85.30,0.82,37.70,11.00,0.0,0.000000,129.61,19.01
6,2024-01-01 01:30:00.002,53.137273,5.468732,84.29,0.00,32.84,10.90,0.0,0.000000,19.00,8.91
7,2024-01-01 01:45:00.002,44.362143,0.000000,0.00,0.00,32.80,0.00,0.0,0.000000,0.00,0.00
8,2024-01-01 02:00:00.002,71.124595,43.961622,89.00,0.00,72.84,0.00,0.0,0.000000,117.02,61.90
9,2024-01-01 02:15:00.003,72.431033,41.442833,86.76,0.00,66.83,5.20,0.0,0.000000,142.78,34.19


## 2. Day-Ahead Market Analysis

The day-ahead market is the primary energy trading market, with 15-minute resolution pricing in EUR/MWh.

### 2.1 Multi-Country Price Comparison

In [4]:
# Plot day-ahead price distribution across all countries
fig = plot_day_ahead_distribution(tables['day_ahead'])
fig.update_layout(height=500, title='Day-Ahead Price Distribution by Country (2024)')
fig.show()

### 2.2 Germany (DE) - Detailed Analysis

In [5]:
# Germany price time series (full year)
fig = plot_price_time_series_mckinsey(
    tables, 
    country='DE', 
    time_range='full',
    markets=['day_ahead']  # Only day-ahead for clarity
)
fig.update_layout(height=500)
fig.show()

In [6]:
# Germany price distribution with KDE
fig = plot_da_price_distribution_mckinsey(
    tables['day_ahead'], 
    country='DE',
    bins=50
)
fig.show()

In [10]:
tables.keys()

dict_keys(['afrr_capacity', 'afrr_energy', 'day_ahead', 'fcr'])

In [12]:
# Germany price heatmap (Hour x Month pattern)
fig = plot_da_price_heatmap_mckinsey(
    tables['day_ahead'],
    country='HU'
)
fig.show()

In [ ]:
# Germany price statistics table
stats_de = calculate_price_statistics_mckinsey(
    tables, 
    country='DE', 
    market='day_ahead'
)

fig = plot_price_statistics_mckinsey(stats_de, 'DE', 'Day-Ahead')
fig.show()

# Also show as DataFrame
display(stats_de)

### 2.3 Quarterly Analysis (Germany)

In [ ]:
# Compare all quarters for Germany
from plotly.subplots import make_subplots

quarters = ['Q1', 'Q2', 'Q3', 'Q4']
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Q1 (Jan-Mar)', 'Q2 (Apr-Jun)', 'Q3 (Jul-Sep)', 'Q4 (Oct-Dec)']
)

for idx, quarter in enumerate(quarters):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    # Get quarterly data
    quarter_map = {'Q1': [1,2,3], 'Q2': [4,5,6], 'Q3': [7,8,9], 'Q4': [10,11,12]}
    months = quarter_map[quarter]
    df_q = tables['day_ahead'][tables['day_ahead']['timestamp'].dt.month.isin(months)]
    
    country_col = 'DE_LU'
    fig.add_trace(
        go.Scatter(
            x=df_q['timestamp'],
            y=df_q[country_col],
            mode='lines',
            name=quarter,
            line=dict(color=COUNTRY_COLORS['DE'], width=1),
            showlegend=(idx==0)
        ),
        row=row, col=col
    )

fig.update_xaxes(title_text='Date', row=2)
fig.update_yaxes(title_text='Price (EUR/MWh)')
fig = apply_mckinsey_style(fig, title='Germany Day-Ahead Prices - Quarterly Breakdown')
fig.update_layout(height=600, showlegend=True)
fig.show()

### 2.4 Cross-Country Comparison

In [ ]:
# Compare statistics across all countries
countries = ['DE', 'AT', 'CH', 'HU', 'CZ']
stats_all = []

for country in countries:
    stats = calculate_price_statistics_mckinsey(
        tables, 
        country=country, 
        market='day_ahead'
    )
    stats['Country'] = country
    stats_all.append(stats)

# Combine into single DataFrame
df_stats_combined = pd.concat(stats_all, ignore_index=True)

# Pivot for better visualization
df_pivot = df_stats_combined.pivot(index='Metric', columns='Country', values='Value')

print("Day-Ahead Price Statistics - All Countries:\n")
display(df_pivot)

In [ ]:
# Visualize mean prices across countries
mean_prices = []
for country in countries:
    col = 'DE_LU' if country == 'DE' else country
    mean_prices.append(tables['day_ahead'][col].mean())

fig = go.Figure()
fig.add_trace(go.Bar(
    x=countries,
    y=mean_prices,
    marker_color=[get_country_color(c) for c in countries],
    text=[f'{p:.1f}' for p in mean_prices],
    textposition='outside'
))

fig = apply_mckinsey_style(fig, title='Average Day-Ahead Prices by Country (2024)')
fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Mean Price (EUR/MWh)',
    height=400,
    showlegend=False
)
fig.show()

## 3. FCR Capacity Market Analysis

Frequency Containment Reserve (FCR) capacity market with 4-hour block pricing in EUR/MW.

In [ ]:
# FCR price distribution across countries
fig = plot_fcr_distribution(tables['fcr'])
fig.update_layout(height=500, title='FCR Capacity Price Distribution by Country (2024)')
fig.show()

In [ ]:
# FCR time series for Germany
fig = plot_price_time_series_mckinsey(
    tables, 
    country='DE', 
    time_range='full',
    markets=['fcr']
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# FCR statistics for Germany
stats_fcr_de = calculate_price_statistics_mckinsey(
    tables, 
    country='DE', 
    market='fcr'
)

fig = plot_price_statistics_mckinsey(stats_fcr_de, 'DE', 'FCR Capacity')
fig.show()

## 4. aFRR Capacity Market Analysis

Automatic Frequency Restoration Reserve capacity market with positive and negative directions.

In [ ]:
# aFRR capacity distribution (Pos vs Neg)
fig = plot_afrr_distribution(tables['afrr_capacity'])
fig.update_layout(height=500, title='aFRR Capacity Price Distribution (Positive vs Negative)')
fig.show()

In [ ]:
# aFRR capacity time series (Germany - both directions)
fig = plot_price_time_series_mckinsey(
    tables, 
    country='DE', 
    time_range='full',
    markets=['afrr_capacity']
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Compare Positive vs Negative aFRR capacity for Germany
df_afrr_cap = tables['afrr_capacity']

fig = go.Figure()

fig.add_trace(go.Box(
    y=df_afrr_cap['DE_Pos'],
    name='Positive',
    marker_color=MCKINSEY_COLORS['positive']
))

fig.add_trace(go.Box(
    y=df_afrr_cap['DE_Neg'],
    name='Negative',
    marker_color=MCKINSEY_COLORS['negative']
))

fig = apply_mckinsey_style(fig, title='Germany aFRR Capacity - Positive vs Negative Distribution')
fig.update_layout(
    yaxis_title='Price (EUR/MW)',
    height=400,
    showlegend=True
)
fig.show()

## 5. aFRR Energy Market Analysis (NEW in Phase 2)

aFRR energy activation prices with 15-minute resolution in EUR/MWh.

In [ ]:
# aFRR energy distribution (Pos vs Neg)
fig = plot_afrr_distribution(tables['afrr_energy'])
fig.update_layout(height=500, title='aFRR Energy Price Distribution (Positive vs Negative)')
fig.show()

In [ ]:
# aFRR energy time series (Germany)
fig = plot_price_time_series_mckinsey(
    tables, 
    country='DE', 
    time_range='Q1',  # Use Q1 for clearer visualization
    markets=['afrr_energy']
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Check for zero activation prices (common when no activation occurs)
df_afrr_energy = tables['afrr_energy']

zero_analysis = []
for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    for direction in ['Pos', 'Neg']:
        col = f'{country}_{direction}'
        if col in df_afrr_energy.columns:
            total = len(df_afrr_energy)
            zeros = (df_afrr_energy[col] == 0).sum()
            zero_pct = zeros / total * 100
            mean_non_zero = df_afrr_energy[df_afrr_energy[col] != 0][col].mean()
            
            zero_analysis.append({
                'Country': country,
                'Direction': direction,
                'Zero Count': zeros,
                'Zero %': f'{zero_pct:.1f}%',
                'Mean (Non-Zero)': f'{mean_non_zero:.2f}' if not pd.isna(mean_non_zero) else 'N/A'
            })

df_zero_analysis = pd.DataFrame(zero_analysis)

print("aFRR Energy Price - Zero Activation Analysis:\n")
print("(High % of zeros indicates periods with no reserve activation)\n")
display(df_zero_analysis)

## 6. Multi-Market Comparison (Germany)

In [ ]:
# All markets for Germany in one view (Q1 only for clarity)
fig = plot_price_time_series_mckinsey(
    tables, 
    country='DE', 
    time_range='Q1',
    markets=['day_ahead', 'fcr', 'afrr_capacity', 'afrr_energy']
)
fig.update_layout(height=600)
fig.show()

## 7. Price Volatility Analysis

In [ ]:
# Calculate daily volatility (standard deviation) for day-ahead prices
df_da = tables['day_ahead'].copy()
df_da['date'] = df_da['timestamp'].dt.date

volatility_data = []
for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    col = 'DE_LU' if country == 'DE' else country
    daily_vol = df_da.groupby('date')[col].std()
    volatility_data.append({
        'country': country,
        'mean_daily_volatility': daily_vol.mean(),
        'max_daily_volatility': daily_vol.max(),
        'dates': pd.to_datetime(daily_vol.index),
        'volatility': daily_vol.values
    })

# Plot volatility over time
fig = go.Figure()

for data in volatility_data:
    fig.add_trace(go.Scatter(
        x=data['dates'],
        y=data['volatility'],
        mode='lines',
        name=data['country'],
        line=dict(color=get_country_color(data['country']), width=1.5)
    ))

fig = apply_mckinsey_style(fig, title='Day-Ahead Price Volatility (Daily Std Dev)')
fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Daily Standard Deviation (EUR/MWh)',
    height=500,
    hovermode='x unified'
)
fig.show()

In [ ]:
# Volatility comparison bar chart
countries_vol = [d['country'] for d in volatility_data]
mean_vols = [d['mean_daily_volatility'] for d in volatility_data]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=countries_vol,
    y=mean_vols,
    marker_color=[get_country_color(c) for c in countries_vol],
    text=[f'{v:.1f}' for v in mean_vols],
    textposition='outside'
))

fig = apply_mckinsey_style(fig, title='Average Daily Price Volatility by Country')
fig.update_layout(
    xaxis_title='Country',
    yaxis_title='Mean Daily Std Dev (EUR/MWh)',
    height=400,
    showlegend=False
)
fig.show()

## 8. Price Correlation Analysis

In [ ]:
# Calculate correlation matrix for day-ahead prices
df_da = tables['day_ahead'].copy()

# Rename DE_LU to DE for cleaner display
df_da = df_da.rename(columns={'DE_LU': 'DE'})

price_cols = ['DE', 'AT', 'CH', 'HU', 'CZ']
corr_matrix = df_da[price_cols].corr()

print("Day-Ahead Price Correlation Matrix:\n")
display(corr_matrix.round(3))

In [ ]:
# Visualize correlation matrix as heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    zmin=-1,
    zmax=1,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 12},
    colorbar=dict(title='Correlation')
))

fig = apply_mckinsey_style(fig, title='Day-Ahead Price Correlation Matrix')
fig.update_layout(
    height=500,
    xaxis_title='Country',
    yaxis_title='Country'
)
fig.show()

## 9. Summary Report

In [ ]:
# Generate comprehensive summary report
print("="*80)
print("TECHARENA 2025 PHASE 2 - MARKET ANALYSIS SUMMARY REPORT")
print("="*80)

print(f"\n📅 Analysis Period: {metadata['date_range']['start'][:10]} to {metadata['date_range']['end'][:10]}")
print(f"📊 Total Data Points: {len(tables['day_ahead']):,} timestamps (15-min resolution)")

print("\n" + "="*80)
print("DAY-AHEAD MARKET SUMMARY")
print("="*80)

for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    col = 'DE_LU' if country == 'DE' else country
    prices = tables['day_ahead'][col]
    print(f"\n{country}:")
    print(f"  Mean Price:      {prices.mean():8.2f} EUR/MWh")
    print(f"  Median Price:    {prices.median():8.2f} EUR/MWh")
    print(f"  Std Dev:         {prices.std():8.2f} EUR/MWh")
    print(f"  Price Range:     [{prices.min():6.2f}, {prices.max():6.2f}] EUR/MWh")
    print(f"  Negative Hours:  {(prices < 0).sum():8d} ({(prices < 0).sum() / len(prices) * 100:.1f}%)")

print("\n" + "="*80)
print("FCR CAPACITY MARKET SUMMARY")
print("="*80)

for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    if country in tables['fcr'].columns:
        prices = tables['fcr'][country]
        print(f"\n{country}:")
        print(f"  Mean Price:      {prices.mean():8.2f} EUR/MW")
        print(f"  Median Price:    {prices.median():8.2f} EUR/MW")
        print(f"  Price Range:     [{prices.min():6.2f}, {prices.max():6.2f}] EUR/MW")

print("\n" + "="*80)
print("aFRR ENERGY MARKET SUMMARY (NEW)")
print("="*80)

for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    print(f"\n{country}:")
    for direction in ['Pos', 'Neg']:
        col = f'{country}_{direction}'
        if col in tables['afrr_energy'].columns:
            prices = tables['afrr_energy'][col]
            non_zero = prices[prices != 0]
            zero_pct = (prices == 0).sum() / len(prices) * 100
            print(f"  {direction}:")
            print(f"    Zero Activation: {zero_pct:6.1f}%")
            if len(non_zero) > 0:
                print(f"    Mean (non-zero): {non_zero.mean():8.2f} EUR/MWh")
                print(f"    Max Price:       {non_zero.max():8.2f} EUR/MWh")

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

# Find country with highest/lowest mean day-ahead price
mean_prices_dict = {}
for country in ['DE', 'AT', 'CH', 'HU', 'CZ']:
    col = 'DE_LU' if country == 'DE' else country
    mean_prices_dict[country] = tables['day_ahead'][col].mean()

highest_price_country = max(mean_prices_dict, key=mean_prices_dict.get)
lowest_price_country = min(mean_prices_dict, key=mean_prices_dict.get)

print(f"\n✓ Highest mean DA price: {highest_price_country} ({mean_prices_dict[highest_price_country]:.2f} EUR/MWh)")
print(f"✓ Lowest mean DA price:  {lowest_price_country} ({mean_prices_dict[lowest_price_country]:.2f} EUR/MWh)")

# Price correlation
corr_matrix = df_da[['DE', 'AT', 'CH', 'HU', 'CZ']].corr()
min_corr = corr_matrix.min().min()
max_corr = corr_matrix[corr_matrix < 1].max().max()

print(f"\n✓ Price correlation range: [{min_corr:.2f}, {max_corr:.2f}]")
print(f"  → Strong price coupling observed across European markets")

print("\n" + "="*80)
print("END OF REPORT")
print("="*80)

## 10. Export Analysis Results

In [ ]:
# Optional: Export key figures to HTML for sharing
output_dir = Path('analysis_outputs')
output_dir.mkdir(exist_ok=True)

print("Exporting figures to HTML...\n")

# Example: Export Germany heatmap
fig = plot_da_price_heatmap_mckinsey(tables['day_ahead'], 'DE')
fig.write_html(output_dir / 'de_price_heatmap.html')
print("✓ Exported: de_price_heatmap.html")

# Export multi-market comparison
fig = plot_price_time_series_mckinsey(
    tables, country='DE', time_range='Q1',
    markets=['day_ahead', 'fcr', 'afrr_capacity', 'afrr_energy']
)
fig.write_html(output_dir / 'de_all_markets_q1.html')
print("✓ Exported: de_all_markets_q1.html")

print(f"\n📁 All exports saved to: {output_dir.absolute()}")

---

## Analysis Complete

This notebook provides a comprehensive overview of the Phase 2 electricity market data using professional McKinsey-style visualizations. 

**Key Findings:**
1. Day-ahead prices show strong seasonal and hourly patterns
2. High price correlation across neighboring European markets
3. aFRR energy prices have significant zero-activation periods
4. FCR capacity prices show lower volatility than energy markets

**Next Steps:**
- Use these insights for battery optimization strategy
- Identify arbitrage opportunities in multi-market participation
- Analyze seasonal patterns for long-term investment decisions